# H2 test
This is a reference test for the implementation same calculation

In [1]:
from pyscf import gto, scf, mp
from py_mods.src.SCF.external import UHF_context_from_pyscf
from py_mods.src.SCF.CSUHF import CS_UHF
from py_mods.src.MP2.CSMP2 import CS_MP2
import numpy as np

mol = gto.M(
    atom="H 0 0 0; H 0 0 1.4;",
    unit="Bohr",
    basis={
        "H": gto.basis.parse(
            f"""
        # This is the standard STO-3G definition
        # Exponent    Contraction Coeff
        H S
        3.42525091    0.15432897
        0.62391373    0.53532814
        0.16885540    0.44463454
        H P
        3.42525091    0.15432897
        0.62391373    0.53532814
        0.16885540    0.44463454
    """
        )
    },
    cart=False,
    spin=0,
)

mol.build()

mf = scf.UHF(mol)
mf.conv_tol = 1e-14
mf.conv_tol_grad = 1e-14

mf.kernel()
mymp = mp.UMP2(mf).run()
print(mf.energy_elec()[0])
print(mf.energy_elec()[0] + mymp.e_corr)


# implementation and calculation remember that we had specified bohr in the previous
dist = 1.4 * 0.529177210544 
pyscf_args = {"atom": f"H 0 0 0; H 0 0 {dist}", "spin": 0, "charge": 0}

RHF_cxt = UHF_context_from_pyscf(
    **pyscf_args,
    basis=gto.basis.parse(
            f"""
        # This is the standard STO-3G definition
        # Exponent    Contraction Coeff
        H S
        3.42525091    0.15432897
        0.62391373    0.53532814
        0.16885540    0.44463454
        H P
        3.42525091    0.15432897
        0.62391373    0.53532814
        0.16885540    0.44463454
    """),
    cart=True,
)

RHF_cxt.verbose = True

RHF_res = CS_UHF(RHF_cxt)
mp_results = CS_MP2(RHF_res)

print(
    f"Final RHF Energy PySCF: {mf.energy_elec()[0]}, Implementation: {RHF_res.E_UHF.real}. Difference {mf.energy_elec()[0]-RHF_res.E_UHF.real:4e}"
)
print(
    f"Final MP2 Energy PySCF: {mf.energy_elec()[0] + mymp.e_corr}, Implementation: {mp_results.E_MP2.real}. Difference { (mf.energy_elec()[0] + mymp.e_corr) - mp_results.E_MP2.real:4e}"
)

SCF not converged.
SCF energy = -1.1206321341139 after 50 cycles  <S^2> = 4.4408921e-16  2S+1 = 1
E(UMP2) = -1.14393597971305  E_corr = -0.0233038455991475
E(SCS-UMP2) = -1.14859674883288  E_corr = -0.027964614718977
-1.8349178483996178
-1.8582216939987652


Alpha occupation:  [1 0 0 0 0 0 0 0]
Beta  occupation:  [1 0 0 0 0 0 0 0]
--------------------------------------------------------------------------------------------------------------------------------
|   Iter     |                   E_iter                      |                   Delta_e                   |      norm(e_i)      |
--------------------------------------------------------------------------------------------------------------------------------
    1            0.0000000000000000+0.0000000000000000j            0.0000000000000000+0.0000000000000000j     0.0000E+00
    2           -1.8234502846672951+0.0000000000000000j           -1.8234502846672951+0.0000000000000000j     1.4953E-01
    3           -1.8346342852596813+